# 💉 Gemma-4 QLoRA — NOTEBOOK A: TRAINING (Phase A)

**Phần 1/2 của pipeline Gemma-4 XAI** · Taxonomy v3 · HUPH 2026

---

## 🔗 LIÊN THÔNG 2 NOTEBOOK

| Notebook | Vai trò | Output |
|---|---|---|
| **A — TRAINING** (notebook này) | Fine-tune QLoRA → push model | HF Hub: `quynhphuong1209/gemma-4-E4B-unsloth-vaccine-xai` |
| **B — INFERENCE** | Load model từ HF → đánh giá Gold Test | `gemma_v3_results.json` |

### Quy trình
1. Chạy **Notebook A** (notebook này) → train xong tự động push lên HF Hub
2. Chạy **Notebook B** → tự động pull model mới nhất từ HF Hub → inference + eval
3. Notebook B xuất `gemma_v3_results.json` → dùng cho notebook báo cáo benchmark

> ⚙️ **Tiết kiệm quota:** Sau khi train 1 lần, mọi thử nghiệm parser/prompt
> chỉ cần chạy lại Notebook B (~1h) — KHÔNG train lại (~2h).

⏱️ Ước tính Notebook A: ~1.5-2h (3 epoch + early-stop)


In [ ]:
# [CELL 1] Cài đặt Unsloth — bản vá: hiện lỗi thật + DỪNG nếu fail
import subprocess, sys, importlib

def pip_install(args, desc):
    """Chạy pip, IN TOÀN BỘ output (không giấu lỗi). Trả về returncode."""
    print(f"🏃 {desc}")
    print(f"   $ pip install {' '.join(args)}")
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", *args],
        capture_output=True, text=True
    )
    if proc.returncode != 0:
        print("   ❌ STDOUT:", proc.stdout[-1500:])
        print("   ❌ STDERR:", proc.stderr[-1500:])
    else:
        print("   ✅ OK")
    return proc.returncode

print("📦 Bước 1/2 — Cài Unsloth (thử git mới nhất, fallback pip ổn định)")

# Thử 1: bản git mới nhất (fix Gemma-4). Nếu fail → thử bản pip ổn định.
rc = pip_install(
    ['--upgrade', '--no-cache-dir',
     'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git'],
    "Thử bản git mới nhất"
)
if rc != 0:
    print("\n⚠️  Bản git fail → fallback bản pip ổn định")
    rc = pip_install(['--upgrade', '--no-cache-dir', 'unsloth'],
                     "Fallback: unsloth ổn định từ PyPI")
if rc != 0:
    print("\n⚠️  Vẫn fail → thử cài thủ công deps + unsloth --no-deps")
    pip_install(['--no-cache-dir',
                 'bitsandbytes', 'accelerate', 'peft', 'trl',
                 'triton', 'xformers', 'unsloth_zoo'],
                "Cài deps thủ công")
    rc = pip_install(['--no-cache-dir', '--no-deps', 'unsloth'],
                     "Cài unsloth --no-deps")

print("\n📦 Bước 2/2 — Xác minh import")

# Numpy check (Unsloth cần numpy>=2.0; Kaggle có sẵn 2.0.2)
try:
    import numpy as np
    print(f"   ✅ Numpy: {np.__version__}")
except Exception as e:
    print(f"   ⚠️  Numpy: {e}")

# GPU check
subprocess.run("nvidia-smi -L", shell=True)

# CRITICAL: import phải thành công, NẾU KHÔNG → raise để DỪNG notebook ngay
# (tránh chạy tiếp tới Cell 2 rồi crash khó hiểu)
try:
    importlib.invalidate_caches()
    from unsloth import FastModel  # noqa: F401
    print("   ✅ Unsloth FastModel import OK — sẵn sàng")
except Exception as e:
    raise RuntimeError(
        f"❌ CÀI UNSLOTH THẤT BẠI: {e}\n"
        "→ Xem STDERR phía trên để biết nguyên nhân.\n"
        "→ Kiểm tra: Kaggle Settings > Internet phải BẬT (ON).\n"
        "→ Kiểm tra: Accelerator phải là GPU (T4/P100), không phải None/CPU."
    )


In [ ]:
# [CELL 2] Imports & GPU diagnostics
import os, gc, json, glob, datetime, shutil
import torch
from huggingface_hub import login, HfApi
from datasets import load_dataset

# Unsloth — nếu lỗi ở đây nghĩa là Cell 1 chưa chạy hoặc cài fail
try:
    from unsloth import FastModel
    from unsloth.chat_templates import get_chat_template, train_on_responses_only
except ModuleNotFoundError as e:
    raise RuntimeError(
        f"❌ {e}\n"
        "→ Cell 1 (cài Unsloth) CHƯA chạy hoặc đã FAIL.\n"
        "→ Chạy lại Cell 1 và đọc kỹ output. Internet phải BẬT."
    )

from trl import SFTTrainer, SFTConfig
from IPython.display import FileLink, display

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1e9
    print(f"🖥️  GPU: {props.name}")
    print(f"💾 VRAM: {total_gb:.1f} GB")
    print(f"🔢 CUDA: {torch.version.cuda}")
    print(f"⚡ BF16: {'✅' if torch.cuda.is_bf16_supported() else '❌ (dùng FP16)'}")
    if total_gb < 14:
        print("⚠️  CẢNH BÁO: VRAM < 14GB — có thể OOM khi train")
else:
    raise RuntimeError("❌ Không tìm thấy GPU — bật GPU T4 trong Kaggle Settings")

print("✅ Imports OK")


### Bước 2: Cấu hình tập trung

In [ ]:
# [CELL 3] CẤU HÌNH TẬP TRUNG — Taxonomy v3 (TRAINING — Notebook A)

def _find(patterns, fallback):
    for p in patterns:
        hits = glob.glob(p, recursive=True)
        if hits: return hits[0]
    return fallback

TRAIN_PATH = _find(
    ["/kaggle/input/**/train_v2_seg_v3.jsonl",
     "/kaggle/input/**/train_v2_seg.jsonl"],
    "/kaggle/input/vaccinenlp-clean-data/05_model_ready/train_v2_seg_v3.jsonl")
TEST_PATH  = _find(
    ["/kaggle/input/**/benchmark_test_set_v3.jsonl",
     "/kaggle/input/**/benchmark_test_set.jsonl"],
    "/kaggle/input/vaccinenlp-clean-data/03_processed/benchmark_test_set_v3.jsonl")

MODELS_SAVE_DIR = "/kaggle/working/gemma_qlora_xai"
INFERENCE_OUT   = "/kaggle/working/gemma_inference_results_v3.jsonl"
RESULTS_JSON    = "/kaggle/working/gemma_v3_results.json"
HF_REPO_ID      = "hung2903/gemma-4-E4B-unsloth-vaccine-xai"

# --- Model ---
BASE_MODEL   = "unsloth/gemma-4-E4B-it"
MAX_SEQ_LEN  = 1280            # đủ cho reasoning + block kết quả, không phí VRAM
LOAD_IN_4BIT = True

# --- LoRA (NÂNG CẤP: r=32 tăng dung lượng học cho dataset 1.6k mẫu) ---
LORA_R       = 32
LORA_ALPHA   = 32              # alpha = r → scaling 1.0, ổn định
LORA_DROPOUT = 0.10            # tăng chống overfit (train loss về 0 quá nhanh)

# --- Training (NÂNG CẤP: 4 epoch + early stopping chọn checkpoint tốt nhất) ---
BATCH_SIZE    = 2
GRAD_ACCUM    = 8              # effective batch = 16 → gradient ổn định hơn
WARMUP_RATIO  = 0.05
TRAIN_EPOCHS  = 3              # curve cho thấy best≈epoch 2; early-stop sẽ dừng
LEARNING_RATE = 1e-4           # giảm thêm: train loss tụt về 0 sau epoch 1 = LR quá cao
WEIGHT_DECAY  = 0.01
LR_SCHEDULER  = "cosine"
SEED          = 42
EARLY_STOP_PATIENCE = 1        # eval_loss xấu đi ngay sau best → dừng sớm

# --- (Inference params nằm ở Notebook B, không cần ở đây) ---

# ── Label Maps — Taxonomy v3 CANONICAL ───────────────────────────────────
MISINFO_MAP   = {0: 'Tin gia',   1: 'Chinh xac'}
STANCE_MAP    = {0: 'Ung ho',    1: 'Phan doi',  2: 'Trung lap'}
SENTIMENT_MAP = {0: 'Tieu cuc',  1: 'Trung tinh', 2: 'Tich cuc'}
HC_MISINFO    = {0: 'Tin giả',   1: 'Chính xác'}
HC_STANCE     = {0: 'Ủng hộ',    1: 'Phản đối',  2: 'Trung lập'}
HC_SENTIMENT  = {0: 'Tiêu cực',  1: 'Trung tính', 2: 'Tích cực'}
N_CLASSES     = {'misinfo': 2, 'stance': 3, 'sentiment': 3}

os.makedirs(MODELS_SAVE_DIR, exist_ok=True)
print("✅ Config v3 POWER MODE loaded")
print(f"   LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"   Epochs={TRAIN_EPOCHS}, eff.batch={BATCH_SIZE*GRAD_ACCUM}, LR={LEARNING_RATE}")
print(f"   Early-stop patience={EARLY_STOP_PATIENCE}")
print(f"   Train: {TRAIN_PATH}")
print(f"   Test : {TEST_PATH}")


### Bước 3: Load Gemma-4 + LoRA Config

In [ ]:
# [CELL 4] Load Gemma-4 E4B + LoRA adapter (r=32 POWER)
model, tokenizer = FastModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded: {BASE_MODEL}")
print(f"   Total params    : {total_params/1e6:.1f}M")
print(f"   Trainable params: {trainable_params/1e6:.1f}M ({100*trainable_params/total_params:.2f}%)")


### Bước 4: Chuẩn bị Dataset & Format Prompt

In [ ]:
# [CELL 5] Dataset prep — ANSWER-FIRST format (KẾT QUẢ trước, Lý luận sau)
# Lựa chọn 1: đảo thứ tự để nhãn sinh NGAY (~30 tok, không bị cắt),
# reasoning đặt sau được phép dài tùy ý → parse rate >90% + XAI chất lượng cao
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

MAX_REASONING_CHARS = 600   # llm_thought 400-900 chars; 600 giữ ngữ cảnh tốt

def _extract_reasoning(row):
    raw = (row.get('llm_thought') or row.get('llm_raw_output') or '').strip()
    if not raw:
        return 'Phân tích dựa trên nội dung và ngữ cảnh ngôn ngữ.'
    for marker in ['Kết quả:', 'Kết luận:', '=== KẾT QUẢ', 'Result:']:
        if marker in raw:
            raw = raw.split(marker)[0].strip()
    for pre in ['Reasoning:', 'Lý luận:', 'Phân tích:']:
        if raw.startswith(pre):
            raw = raw[len(pre):].strip()
    if len(raw) > MAX_REASONING_CHARS:
        cut = raw[:MAX_REASONING_CHARS]
        for sep in ['. ', '.\n', '? ', '! ']:
            if sep in cut:
                cut = cut.rsplit(sep, 1)[0] + sep.strip()
                break
        raw = cut
    return raw if raw else 'Phân tích dựa trên nội dung và ngữ cảnh ngôn ngữ.'

def format_prompt(row):
    text      = (row.get('text_cleaned') or row.get('text') or '')[:600]
    reasoning = _extract_reasoning(row)

    ids = row.get('standardized_ids')
    if not ids or not isinstance(ids, (list, tuple)) or len(ids) < 3:
        ids = [1, 2, 1]
    misinfo_label = MISINFO_MAP.get(int(ids[0]),  'Chinh xac')
    stance_label  = STANCE_MAP.get(int(ids[1]),   'Trung lap')
    senti_label   = SENTIMENT_MAP.get(int(ids[2]),'Trung tinh')

    convo = [
        {
            "role": "user",
            "content": (
                f'Phân tích nội dung sau về vaccine:\n\n"{text}"\n\n'
                'Trả lời theo ĐÚNG cấu trúc: nêu KẾT QUẢ trước, GIẢI THÍCH sau.\n'
                '=== KẾT QUẢ ===\n'
                '- Misinformation: <Tin gia HOẶC Chinh xac>\n'
                '- Stance: <Ung ho HOẶC Phan doi HOẶC Trung lap>\n'
                '- Sentiment: <Tieu cuc HOẶC Trung tinh HOẶC Tich cuc>\n'
                '=== GIẢI THÍCH ===\n'
                '<lý luận chi tiết bằng tiếng Việt>'
            )
        },
        {
            "role": "assistant",
            "content": (
                # ANSWER-FIRST: nhãn sinh NGAY → không bao giờ bị cắt
                f'=== KẾT QUẢ ===\n'
                f'- Misinformation: {misinfo_label}\n'
                f'- Stance: {stance_label}\n'
                f'- Sentiment: {senti_label}\n'
                f'=== GIẢI THÍCH ===\n'
                f'{reasoning}'
            )
        }
    ]
    return tokenizer.apply_chat_template(convo, tokenize=False)

raw_ds = load_dataset("json", data_files={"train": TRAIN_PATH}, split="train")

def _mis_id(r):
    ids = r.get('standardized_ids')
    return int(ids[0]) if ids and len(ids) >= 3 else 1

split     = raw_ds.train_test_split(test_size=0.1, seed=SEED)
train_raw = split["train"]
eval_raw  = split["test"]
minority  = train_raw.filter(lambda r: _mis_id(r) == 0)
from datasets import concatenate_datasets
train_balanced = concatenate_datasets([train_raw, minority]).shuffle(seed=SEED)

train_ds = train_balanced.map(lambda r: {"text": format_prompt(r)}, num_proc=2)
eval_ds  = eval_raw.map(      lambda r: {"text": format_prompt(r)}, num_proc=2)

_txt_tok = getattr(tokenizer, "tokenizer", tokenizer)
_lens, _kq_first, _real = [], 0, 0
_fb = 'Phân tích dựa trên nội dung và ngữ cảnh ngôn ngữ.'
for i in range(min(200, len(train_ds))):
    txt = train_ds[i]['text']
    _lens.append(len(_txt_tok(txt)['input_ids']))
    # KẾT QUẢ phải xuất hiện TRƯỚC GIẢI THÍCH (answer-first)
    if '=== KẾT QUẢ ===' in txt and '=== GIẢI THÍCH ===' in txt:
        if txt.index('=== KẾT QUẢ ===') < txt.index('=== GIẢI THÍCH ==='):
            _kq_first += 1
    if _fb not in txt:
        _real += 1
import numpy as _np
_arr = _np.array(_lens)
print(f"✅ Dataset ready — ANSWER-FIRST format (Lựa chọn 1)")
print(f"   Train: {len(train_raw)} → {len(train_ds)} (oversample Tin giả)")
print(f"   Eval : {len(eval_ds)}")
print(f"   Token length: mean={_arr.mean():.0f}, p95={_np.percentile(_arr,95):.0f}, "
      f"max={_arr.max()}, MAX_SEQ_LEN={MAX_SEQ_LEN}")
print(f"   KẾT QUẢ đứng TRƯỚC GIẢI THÍCH: {_kq_first}/200")
print(f"   Mẫu có reasoning THỰC        : {_real}/200")
if _kq_first >= 195 and _real >= 150:
    print(f"   ✅ ANSWER-FIRST OK — nhãn sinh đầu, parse rate sẽ cao, reasoning giữ nguyên")
else:
    print(f"   ⚠️  Kiểm tra lại format_prompt")
print(f"\n   Sample (mẫu 0) — 500 ký tự đầu:\n{'-'*50}")
print(train_ds[0]['text'][:500])


### Bước 5: Huấn luyện QLoRA

In [ ]:
# [CELL 6] SFTTrainer + train_on_responses_only + EarlyStopping
from transformers import EarlyStoppingCallback

# Kiểm tra delimiter THỰC TẾ của chat template (tránh bug mask sai)
_probe = tokenizer.apply_chat_template(
    [{"role":"user","content":"X"},{"role":"assistant","content":"Y"}],
    tokenize=False)
print("🔍 Chat template probe (200 chars đầu):")
print(_probe[:200])

# Gemma-4 chuẩn dùng <start_of_turn>user ... <start_of_turn>model
INSTRUCTION_PART = "<start_of_turn>user\n"
RESPONSE_PART    = "<start_of_turn>model\n"
# Một số build Unsloth dùng <|turn>. Auto-detect cho chắc:
if "<|turn>" in _probe:
    INSTRUCTION_PART = "<|turn>user\n"
    RESPONSE_PART    = "<|turn>model\n"
print(f"   → instruction_part = {INSTRUCTION_PART!r}")
print(f"   → response_part    = {RESPONSE_PART!r}")

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args = SFTConfig(
        output_dir                  = MODELS_SAVE_DIR,
        dataset_text_field          = "text",
        max_length                  = MAX_SEQ_LEN,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = 1,
        gradient_accumulation_steps = GRAD_ACCUM,
        eval_accumulation_steps     = 1,
        warmup_ratio                = WARMUP_RATIO,
        num_train_epochs            = TRAIN_EPOCHS,
        learning_rate               = LEARNING_RATE,
        weight_decay                = WEIGHT_DECAY,
        lr_scheduler_type           = LR_SCHEDULER,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        seed                        = SEED,
        report_to                   = "none",
    ),
    callbacks = [EarlyStoppingCallback(
        early_stopping_patience = EARLY_STOP_PATIENCE)],
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = INSTRUCTION_PART,
    response_part    = RESPONSE_PART,
)

# Sanity: kiểm tra mask hoạt động (labels phải có -100 cho phần prompt)
_s = trainer.train_dataset[0]
_lab = _s.get('labels', [])
_masked = sum(1 for x in _lab if x == -100)
print(f"🔍 Label mask check: {_masked}/{len(_lab)} tokens masked (prompt)")
if _masked == 0:
    print("   ⚠️  CẢNH BÁO: 0 tokens masked → delimiter SAI, model sẽ học cả prompt!")
else:
    print(f"   ✅ Mask OK — model chỉ học phần response ({len(_lab)-_masked} tokens)")

print("\n🚀 Bắt đầu huấn luyện (POWER MODE — 4 epoch, early-stop)...")
import time as _t
_t0 = _t.time()
train_result = trainer.train()
_dur = int(_t.time() - _t0)
print(f"\n✅ Huấn luyện hoàn tất trong {_dur//60}m{_dur%60}s")
print(f"   Loss cuối: {train_result.training_loss:.4f}")
print(f"   Best checkpoint đã tự động load (load_best_model_at_end=True)")

trainer.save_model(f"{MODELS_SAVE_DIR}/final_model")
tokenizer.save_pretrained(f"{MODELS_SAVE_DIR}/final_model")
print(f"✅ Best model đã lưu: {MODELS_SAVE_DIR}/final_model")


### Bước 6b: 📈 Training Curves — QLoRA Phase A

In [ ]:
# [CELL 6b] Training Curves — SFTTrainer log_history
import matplotlib.pyplot as plt, numpy as np

log_history = trainer.state.log_history
train_logs = [(x["step"], x["loss"])      for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs  = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

if not train_logs:
    print("No training logs found")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Gemma-4 QLoRA Training Curves (Phase A)", fontsize=14, fontweight="bold")

    train_steps, train_loss = zip(*train_logs)
    ax = axes[0]
    ax.plot(train_steps, train_loss, "b-", lw=1.5, label="Train Loss (step)", alpha=0.8)
    if eval_logs:
        es, el = zip(*eval_logs)
        ax.plot(es, el, "r-s", lw=2, ms=6, label="Eval Loss (epoch)")
    ax.set(xlabel="Step", ylabel="Loss", title="Loss per Step")
    ax.legend(); ax.grid(True, alpha=0.3)

    total_steps  = trainer.state.max_steps
    steps_per_ep = max(1, total_steps // TRAIN_EPOCHS)
    epoch_means, epoch_nums = [], []
    for ep in range(TRAIN_EPOCHS):
        ep_losses = [l for s, l in train_logs if ep*steps_per_ep <= s < (ep+1)*steps_per_ep]
        if ep_losses:
            epoch_means.append(np.mean(ep_losses))
            epoch_nums.append(ep + 1)

    ax = axes[1]
    if epoch_means:
        ax.plot(epoch_nums, epoch_means, "b-o", lw=2, ms=5, label="Train Loss (mean/epoch)")
    if eval_logs:
        es, el = zip(*eval_logs)
        ax.plot(list(range(1, len(el)+1)), el, "r-s", lw=2, ms=6, label="Eval Loss")
        best_ep = int(np.argmin(el)) + 1
        ax.axvline(best_ep, color="green", ls="--", alpha=0.7, label=f"Best Epoch ({best_ep})")
    ax.set(xlabel="Epoch", ylabel="Loss", title="Loss per Epoch")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    curve_path = f"{MODELS_SAVE_DIR}/training_curves.png"
    plt.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {curve_path}")
    print(f"Total steps: {trainer.state.global_step} | Final train loss: {train_loss[-1]:.4f}")
    if eval_logs:
        best = min(zip(*eval_logs), key=lambda x: x[1])
        print(f"Best eval loss: {best[1]:.4f} at step {best[0]}")


### Bước 6: Lưu & Upload lên HuggingFace Hub

In [ ]:
# [CELL 7] 🔗 ĐIỂM LIÊN THÔNG: Push model lên HF Hub cho Notebook B
# ✅ FIX: HF push được wrap trong try/except — Phase B không phụ thuộc vào push thành công

# --- Nén để download ---
print("📦 Nén model để tải về...")
shutil.make_archive('/kaggle/working/gemma_vaccine_model', 'zip', f'{MODELS_SAVE_DIR}/final_model')
print("✅ Đã tạo: /kaggle/working/gemma_vaccine_model.zip")
display(FileLink('/kaggle/working/gemma_vaccine_model.zip'))

# --- Push lên HuggingFace Hub (không bắt buộc) ---
try:
    from kaggle_secrets import UserSecretsClient
    VaccineNLP_TOKEN = UserSecretsClient().get_secret("VaccineNLP")
    if VaccineNLP_TOKEN:
        login(token=VaccineNLP_TOKEN, add_to_git_credential=False)
        api = HfApi()
        api.upload_folder(
            folder_path = f"{MODELS_SAVE_DIR}/final_model",
            repo_id     = HF_REPO_ID,
            repo_type   = "model",
        )
        print(f"✅ Pushed lên HuggingFace: {HF_REPO_ID}")
    else:
        print("⚠️  VaccineNLP rỗng — bỏ qua push")
except Exception as e:
    print(f"⚠️  HF push thất bại (không ảnh hưởng Phase B): {e}")

print(f"\n📁 Model local: {MODELS_SAVE_DIR}/final_model")

---
## ✅ NOTEBOOK A HOÀN TẤT

Model đã được push lên: **`quynhphuong1209/gemma-4-E4B-unsloth-vaccine-xai`**

### Bước tiếp theo
→ Mở **Notebook B (INFERENCE)** và Run All.
Notebook B sẽ tự động pull model vừa train từ HF Hub.

> Không cần làm gì thêm ở đây. Notebook A chỉ chạy 1 lần cho mỗi
> lần thay đổi training. Mọi tinh chỉnh inference → chạy Notebook B.
